## gồm
- adapter
- factory
- strategy (discount rule)
- strategy (integration route)
- pipeline validation

## cấu trúc
```
SkyTransferPayload
    ↓
ValidationPipeline
    ↓
Adapter
    ↓
SkyCrmDiscount
    ↓
Factory
    ↓
DiscountStrategy
    ↓
SyncStrategy
    ↓
RK7 / RefAgent
```

## flow
```
skytransfer
   ↓
adapter
   ↓
pipeline validate
   ↓
factory create handler
   ↓
discount strategy apply
   ↓
sync strategy
   ↓
rk7 / refagent
```

In [1]:
using System;
using System.Collections.Generic;
using System.Threading.Tasks;

// =====================================================
// DOMAIN
// =====================================================

public enum DiscountType
{
    OrderPercent,
    OrderAmount,
    ProductPercent,
    ProductAmount
}

public class SkyTransferPayload
{
    public string Name { get; set; }
    public DiscountType Type { get; set; }
    public double Percent { get; set; }
    public DateTime StartTime { get; set; }
    public DateTime? EndTime { get; set; }
}

public class SkyCrmDiscount
{
    public string Name { get; set; }
    public DiscountType Type { get; set; }
    public double Value { get; set; }
}

public class RK7Payload
{
    public string PromoName { get; set; }
    public double DiscountValue { get; set; }
}

public class RefAgentPayload
{
    public string Code { get; set; }
    public double Value { get; set; }
}

// =====================================================
// ADAPTER PATTERN
// =====================================================

public interface IAdapter<TInput, TOutput>
{
    TOutput Adapt(TInput input);
}

public class SkyTransferToSkyCrmAdapter : IAdapter<SkyTransferPayload, SkyCrmDiscount>
{
    public SkyCrmDiscount Adapt(SkyTransferPayload input)
    {
        return new SkyCrmDiscount
        {
            Name = input.Name,
            Type = input.Type,
            Value = input.Percent
        };
    }
}

public class SkyCrmToRK7Adapter : IAdapter<SkyCrmDiscount, RK7Payload>
{
    public RK7Payload Adapt(SkyCrmDiscount input)
    {
        return new RK7Payload
        {
            PromoName = input.Name,
            DiscountValue = input.Value
        };
    }
}

public class SkyCrmToRefAgentAdapter : IAdapter<SkyCrmDiscount, RefAgentPayload>
{
    public RefAgentPayload Adapt(SkyCrmDiscount input)
    {
        return new RefAgentPayload
        {
            Code = input.Name,
            Value = input.Value
        };
    }
}

// =====================================================
// PIPELINE PATTERN (VALIDATION)
// =====================================================

public interface IPipelineStep<T>
{
    Task Execute(T context);
}

public class PipelineRunner<T>
{
    private readonly IEnumerable<IPipelineStep<T>> steps;

    public PipelineRunner(IEnumerable<IPipelineStep<T>> steps)
    {
        this.steps = steps;
    }

    public async Task Run(T context)
    {
        foreach (var step in steps)
        {
            await step.Execute(context);
        }
    }
}

public class ValidationContext
{
    public SkyTransferPayload Payload { get; set; }
}

public class ValidateNameStep : IPipelineStep<ValidationContext>
{
    public Task Execute(ValidationContext context)
    {
        if (string.IsNullOrEmpty(context.Payload.Name))
            throw new Exception("name required");

        return Task.CompletedTask;
    }
}

public class ValidatePercentStep : IPipelineStep<ValidationContext>
{
    public Task Execute(ValidationContext context)
    {
        if (context.Payload.Percent <= 0 || context.Payload.Percent > 100)
            throw new Exception("invalid percent");

        return Task.CompletedTask;
    }
}

// =====================================================
// STRATEGY PATTERN (DISCOUNT RULE)
// =====================================================

public interface IDiscountStrategy
{
    void Apply(SkyCrmDiscount discount);
}

public class PercentDiscountStrategy : IDiscountStrategy
{
    public void Apply(SkyCrmDiscount discount)
    {
        Console.WriteLine($"apply percent discount {discount.Value}%");
    }
}

public class AmountDiscountStrategy : IDiscountStrategy
{
    public void Apply(SkyCrmDiscount discount)
    {
        Console.WriteLine($"apply amount discount {discount.Value}");
    }
}

// =====================================================
// FACTORY METHOD
// =====================================================

public class DiscountHandlerFactory
{
    public static IDiscountStrategy Create(DiscountType type)
    {
        return type switch
        {
            DiscountType.OrderPercent => new PercentDiscountStrategy(),
            DiscountType.ProductPercent => new PercentDiscountStrategy(),
            DiscountType.OrderAmount => new AmountDiscountStrategy(),
            DiscountType.ProductAmount => new AmountDiscountStrategy(),
            _ => throw new Exception("unknown discount type")
        };
    }
}

// =====================================================
// STRATEGY (SYNC ROUTE)
// =====================================================

public interface ISyncStrategy
{
    Task Sync(SkyCrmDiscount discount);
}

public class RK7SyncStrategy : ISyncStrategy
{
    public Task Sync(SkyCrmDiscount discount)
    {
        var adapter = new SkyCrmToRK7Adapter();
        var payload = adapter.Adapt(discount);

        Console.WriteLine($"sync to rk7 {payload.PromoName}");

        return Task.CompletedTask;
    }
}

public class RefAgentSyncStrategy : ISyncStrategy
{
    public Task Sync(SkyCrmDiscount discount)
    {
        var adapter = new SkyCrmToRefAgentAdapter();
        var payload = adapter.Adapt(discount);

        Console.WriteLine($"sync to refagent {payload.Code}");

        return Task.CompletedTask;
    }
}

// =====================================================
// APPLICATION FLOW
// =====================================================

public class DiscountService
{
    public async Task Process(SkyTransferPayload payload)
    {
        // 1 validate
        var pipeline = new PipelineRunner<ValidationContext>(
            new List<IPipelineStep<ValidationContext>>
            {
                new ValidateNameStep(),
                new ValidatePercentStep()
            });

        await pipeline.Run(new ValidationContext
        {
            Payload = payload
        });

        // 2 adapter
        var adapter = new SkyTransferToSkyCrmAdapter();
        var crmDiscount = adapter.Adapt(payload);

        // 3 factory create strategy
        var strategy = DiscountHandlerFactory.Create(crmDiscount.Type);

        // 4 apply discount rule
        strategy.Apply(crmDiscount);

        // 5 choose sync route
        ISyncStrategy syncStrategy;

        if (crmDiscount.Type == DiscountType.OrderPercent)
            syncStrategy = new RK7SyncStrategy();
        else
            syncStrategy = new RefAgentSyncStrategy();

        // 6 sync
        await syncStrategy.Sync(crmDiscount);
    }
}

// =====================================================
// DEMO
// =====================================================

var payload = new SkyTransferPayload
{
    Name = "promo1",
    Percent = 10,
    Type = DiscountType.OrderPercent,
    StartTime = DateTime.Now
};

var service = new DiscountService();

await service.Process(payload);

apply percent discount 10%
sync to rk7 promo1
